# Пространственное объединение и агрегирование

В предыдущем разделе мы использовали пространственные отношения, чтобы отбирать объекты. Теперь используем их, чтобы **связывать данные из разных слоёв**.

**Пространственное объединение** (_spatial join_) присоединяет к каждому объекту одного слоя атрибуты тех объектов другого слоя, с которыми он находится в заданном пространственном отношении. Например, к каждому театру можно присоединить название округа, в котором он расположен.

После этого данные можно **агрегировать** — перейти от отдельных объектов к пространственным единицам:

- подсчитать количество объектов в каждой единице;
- рассчитать статистику по атрибутам объектов (среднее, медиану и т. д.).

В этом разделе мы подсчитаем число театров в муниципальных округах Санкт-Петербурга и рассчитаем средний год постройки многоквартирных домов по округам.


## 0. Импорт библиотек и подготовка данных

### 0.1 Импорт библиотек


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

### 0.2 Подготовка данных

В этом разделе используются файлы из папки [`data`](https://github.com/bella-mir/spatialAnalysisFall26/tree/main/notebooks/data) нашего репозитория:

- **spb_admin.gpkg** — границы муниципальных округов Санкт-Петербурга (слой `okrug`), в поле `Popul` — численность населения;
- **spb_theaters.csv** — театры Санкт-Петербурга. Источник: [Портал открытых данных Санкт-Петербурга](https://data.gov.spb.ru/)
- **spb_mkd_reforma.csv** — многоквартирные дома (МКД) Санкт-Петербурга. Источник: [Фонд развития территорий](https://витрина.фрт.рф/opendata)


In [ ]:
okrugs = gpd.read_file("../data/spb_admin.gpkg", layer="okrug")

theaters_csv = pd.read_csv("../data/spb_theaters.csv")
theaters_csv = theaters_csv.dropna(subset=["longitude", "latitude"])

theaters = gpd.GeoDataFrame(
    theaters_csv,
    geometry=gpd.points_from_xy(theaters_csv["longitude"], theaters_csv["latitude"]),
    crs=4326
)

m = okrugs.explore(color="gray", style_kwds={"fillOpacity": 0.1})
theaters.explore(m=m, color="red")

## 1. Пространственное объединение

### 1.1. Выполнение объединения

Определим, в каком округе находится каждый театр. Используем функцию `gpd.sjoin()`:

- первый слой — объекты, к которым присоединяем атрибуты (театры);
- второй слой — объекты, атрибуты которых присоединяем (округа);
- `predicate="within"` — театр должен находиться **внутри** округа;
- `how="left"` — сохраняем все театры, даже если для какого-то из них округ не найдётся.

Оба слоя должны быть в одной системе координат:


In [ ]:
okrugs.crs == theaters.crs

In [ ]:
theaters_in_okrug = gpd.sjoin(
    theaters,
    okrugs[["NAME", "geometry"]],
    how="left",
    predicate="within"
)

theaters_in_okrug[["name", "NAME"]].head()

К каждому театру добавилось название округа (`NAME`), а также поле `index_right` — индекс объекта из второго слоя.


### 1.2. Проверка результата

После пространственного объединения всегда стоит проверить две вещи.

**Есть ли объекты, которые не попали ни в один полигон?** Например, точки за пределами территории или на воде. При `how="left"` у них будут пропуски:


In [ ]:
print("Театров без округа:", theaters_in_okrug["NAME"].isna().sum())

**Есть ли объекты, которые попали в несколько полигонов?** Если полигоны перекрываются, а точка лежит в зоне перекрытия, `sjoin` вернёт для неё несколько строк, и при подсчёте она будет учтена дважды:


In [ ]:
print("Театров, попавших в несколько округов:", theaters_in_okrug.index.duplicated().sum())

В нашем случае дублей нет. Если они появятся, решите, к какой единице относить такие объекты, или выберите другой предикат.


## 2. Подсчёт объектов

### 2.1. Агрегирование

Подсчитаем количество театров в каждом округе. Сгруппируем театры по названию округа и посчитаем размер каждой группы:


In [ ]:
theater_counts = (
    theaters_in_okrug
    .groupby("NAME")
    .size()
    .reset_index(name="theater_count")
)

theater_counts.sort_values("theater_count", ascending=False).head()

### 2.2. Присоединение к полигонам

Чтобы показать результат на карте, присоединим таблицу обратно к слою округов. Это обычное табличное объединение по общему полю. Округам без театров присвоим 0:


In [ ]:
okrugs = okrugs.merge(theater_counts, on="NAME", how="left")
okrugs["theater_count"] = okrugs["theater_count"].fillna(0).astype(int)

> Здесь мы объединяем данные по названию округа. В реальных задачах лучше использовать уникальные идентификаторы (коды), чтобы избежать ошибок из-за совпадающих или по-разному записанных названий.


### 2.3. Относительный показатель

Как мы обсуждали в разделе [Картограммы и методы классификации](../module_3/mapping_1.ipynb), для картограмм нужны относительные показатели: абсолютное число театров больше говорит о размере округа, чем о его обеспеченности театрами.

Рассчитаем число театров на 100 тыс. жителей:


In [ ]:
okrugs["theaters_per_100k"] = okrugs["theater_count"] / okrugs["Popul"] * 100_000

okrugs[["NAME", "theater_count", "Popul", "theaters_per_100k"]].sort_values(
    "theaters_per_100k", ascending=False
).head()

In [ ]:
okrugs.explore(
    column="theaters_per_100k",
    cmap="OrRd",
    legend=True,
    legend_kwds={"caption": "Театров на 100 тыс. жителей"}
)

Театры сосредоточены в историческом центре, а в большинстве округов (73 из 111) их нет совсем.

Обратите внимание на округа с маленьким населением: один театр в округе с несколькими тысячами жителей даёт очень высокое значение показателя. При малом знаменателе относительные показатели становятся неустойчивыми — учитывайте это при интерпретации.


## 3. Агрегирование атрибутов

Кроме подсчёта объектов, можно рассчитывать статистику по их **атрибутам**. Рассчитаем средний и медианный год постройки многоквартирных домов в каждом округе.

### 3.1. Подготовка данных

Прочитаем данные о МКД. Координаты уже записаны в полях `lat` и `lon`:


In [ ]:
mkd = pd.read_csv("../data/spb_mkd_reforma.csv", index_col=0, low_memory=False)
mkd = mkd.dropna(subset=["lat", "lon"])

mkd_gdf = gpd.GeoDataFrame(
    mkd,
    geometry=gpd.points_from_xy(mkd["lon"], mkd["lat"]),
    crs=4326
)

print("Домов:", len(mkd_gdf))

In [ ]:
mkd.head()

Чтобы рассчитывать статистику, поле `built_year` (год постройки) должно быть числовым. Проверим его тип:


In [ ]:
mkd_gdf["built_year"].dtype

Поле уже числовое. Но если бы в нём встречались нечисловые записи (например, «нет данных»), pandas прочитал бы его как строки (тип `object`). На этот случай преобразование с помощью `pd.to_numeric()` стоит выполнять всегда: параметр `errors="coerce"` заменяет значения, которые нельзя преобразовать, на `NaN`:


In [ ]:
mkd_gdf["built_year"] = pd.to_numeric(mkd_gdf["built_year"], errors="coerce")

mkd_gdf["built_year"].describe()

В данных встречаются явные ошибки: нулевые значения и даты в будущем. Оставим только реалистичные значения — от основания Санкт-Петербурга до текущего года:


In [ ]:
current_year = pd.Timestamp.now().year

mkd_gdf = mkd_gdf[mkd_gdf["built_year"].between(1703, current_year)]

print(f"Осталось домов: {len(mkd_gdf)}")

### 3.2. Объединение и расчёт статистики

Определим округ каждого дома и одним вызовом `.agg()` рассчитаем несколько показателей сразу:


In [ ]:
mkd_in_okrug = gpd.sjoin(
    mkd_gdf,
    okrugs[["NAME", "geometry"]],
    how="left",
    predicate="within"
)

build_year_stats = (
    mkd_in_okrug
    .groupby("NAME")["built_year"]
    .agg(
        build_year_mean="mean",
        build_year_median="median",
        mkd_count="size",
    )
    .reset_index()
)

okrugs = okrugs.merge(build_year_stats, on="NAME", how="left")

okrugs[["NAME", "mkd_count", "build_year_mean", "build_year_median"]].head()

### 3.3. Сравнение показателей на картах

Сравним средний и медианный год постройки. Как мы знаем из раздела [Сравнение карт и показателей](../module_3/mapping_3.ipynb), сравнивать можно только карты с **общей шкалой**. Для непрерывной шкалы достаточно задать одинаковые границы `vmin` и `vmax`:


In [ ]:
metrics = {
    "build_year_mean": "Средний год постройки",
    "build_year_median": "Медианный год постройки",
}

vmin = okrugs[list(metrics)].min().min()
vmax = okrugs[list(metrics)].max().max()

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

for ax, (column, title) in zip(axes, metrics.items()):
    okrugs.plot(
        column=column,
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        linewidth=0.3,
        edgecolor="white",
        legend=True,
        legend_kwds={"shrink": 0.6},
        missing_kwds={"color": "lightgrey"},
        ax=ax
    )
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

Сравните карты: в каких округах средний и медианный год постройки заметно различаются? Среднее чувствительно к выбросам: несколько очень старых домов среди новых (или наоборот) «смещают» его, а медиана остаётся устойчивой. Поэтому для неравномерно распределённых данных медиана часто лучше описывает «типичный» объект.


## 4. Влияние границ пространственных единиц

Результаты агрегирования зависят от того, **по каким единицам** мы агрегируем. Если провести границы округов иначе или объединить их в районы, значения показателей и картина на карте изменятся, хотя сами объекты останутся теми же.

Эта проблема называется **проблемой изменяемой единицы** (_Modifiable Areal Unit Problem, MAUP_). У неё два проявления:

- **эффект масштаба** — результат зависит от размера единиц (округа или районы);
- **эффект зонирования** — результат зависит от того, как проведены границы единиц одного размера.

Кроме того, административные единицы сильно различаются по площади и форме, что затрудняет их сравнение. Одно из решений — агрегировать данные по **регулярной сетке** одинаковых ячеек. Его мы рассмотрим в следующем разделе.


## Итог

В этом разделе мы:

1. выполнили пространственное объединение с помощью `gpd.sjoin()` и проверили его результат (объекты без пары и дубли);
2. подсчитали число объектов в пространственных единицах и рассчитали относительный показатель;
3. рассчитали статистику по атрибутам объектов, предварительно очистив данные;
4. сравнили показатели на картах с общей шкалой;
5. познакомились с проблемой изменяемой единицы (MAUP).
